# Online Retail SQL Analysis



## Business Questions

This SQL analysis focuses on the following questions:

* How do first-observed and returning customers contribute to monthly revenue and orders?
* How do average order value and basket size differ between one-time and repeat customers?
* How do average order value and basket size differ across the RFM customer segments?
* Which products are purchased by the largest number of repeat customers?




In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

In [2]:
# Locate the cleaned dataset of online retail

data_path = Path(r"clean_online_retail.csv")



print("File found:", data_path.exists())


File found: True


In [3]:
# Load the CSV
retail_sql = pd.read_csv(
    data_path,
    parse_dates=["InvoiceDate"])


In [4]:
# Create the SQLite database connection
conn = sqlite3.connect("online_retail.db")
               

In [5]:
# Load the dataframe into a SQL table

retail_sql.to_sql(
    "online_retail",
    conn,
    if_exists="replace",
    index=False,
    chunksize=10000)

392692

In [6]:
#load the rfm segments dataset
rfm_path = Path(r"rfm_customer_segments.csv")

print("File found:", rfm_path.exists())

File found: True


In [7]:
# Read the CSV

rfm_sql = pd.read_csv(rfm_path)

print("Rows loaded:", len(rfm_sql))
print("Columns:", rfm_sql.columns.tolist())

Rows loaded: 4338
Columns: ['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Segment']


In [8]:
# Load it as a second SQL table

rfm_sql.to_sql(
    "rfm_segments",
    conn,
    if_exists="replace",
    index=False)


4338

In [9]:
# Confirming the online retail table created

pd.read_sql_query(
    """
    SELECT COUNT(*) AS row_count
    FROM online_retail;
    """,
    conn)

,row_count
0,392692


In [10]:
# Confirming the rfm segments table was created correctly

pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT CustomerID) AS customers,
        COUNT(*) - COUNT(DISTINCT CustomerID)
            AS duplicate_customers
    FROM rfm_segments;
    """,
    conn)

,rows,customers,duplicate_customers
0,4338,4338,0


## 1. SQL Data Validation

### Part 1. Data Validation

The cleaned table is validated before analysis to confirm its row count, customer count, order count, revenue and remaining data-quality issues.

In [11]:
validation_query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT CustomerID) AS customers,
    COUNT(DISTINCT InvoiceNo) AS orders,
    ROUND(SUM(Total_price), 2) AS revenue,
    ROUND(
        SUM(Total_price) * 1.0 /
        COUNT(DISTINCT InvoiceNo),
        2
    ) AS average_order_value,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END)
        AS missing_customer_ids,
    SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END)
        AS non_positive_quantities,
    SUM(CASE WHEN UnitPrice <= 0 THEN 1 ELSE 0 END)
        AS non_positive_prices
FROM online_retail;
"""

pd.read_sql_query(validation_query,conn)

,row_count,customers,orders,revenue,average_order_value,missing_customer_ids,non_positive_quantities,non_positive_prices
0,392692,4338,18532,8887208.89,479.56,0,0,0


### Part 2. Duplicate and Date Validation

In [12]:
duplicate_date_query = """
WITH duplicate_groups AS (
    SELECT
        COUNT(*) - 1 AS duplicate_rows
    FROM online_retail
    GROUP BY
        InvoiceNo,
        StockCode,
        Description,
        Quantity,
        InvoiceDate,
        UnitPrice,
        CustomerID,
        Country,
        Total_price
    HAVING COUNT(*) > 1
)

SELECT
    MIN(InvoiceDate) AS first_transaction,
    MAX(InvoiceDate) AS last_transaction,
    COALESCE(
        (SELECT SUM(duplicate_rows) FROM duplicate_groups),
        0
    ) AS exact_duplicate_rows
FROM online_retail;
"""

pd.read_sql_query(duplicate_date_query, conn)

,first_transaction,last_transaction,exact_duplicate_rows
0,2010-12-01 08:26:00,2011-12-09 12:50:00,0


## 2.  Monthly First-Observed vs Returning Customer Contribution

How do first-observed and returning customers contribute to monthly revenue and orders?

A customer is classified as first-observed during their earliest purchase month in the dataset. Activity in later months is classified as returning.


In [13]:
monthly_customer_mix_query = """
WITH customer_first_month AS (
    SELECT
        CustomerID,
        strftime('%Y-%m', MIN(InvoiceDate))
            AS first_observed_month
    FROM online_retail
    GROUP BY CustomerID
),

monthly_customer_activity AS (
    SELECT
        strftime('%Y-%m', InvoiceDate) AS month,
        CustomerID,
        COUNT(DISTINCT InvoiceNo) AS order_count,
        SUM(Total_price) AS customer_revenue
    FROM online_retail
    WHERE DATE(InvoiceDate) < '2011-12-01'
    GROUP BY
        strftime('%Y-%m', InvoiceDate),
        CustomerID
),

classified_activity AS (
    SELECT
        activity.month,
        activity.CustomerID,
        activity.order_count,
        activity.customer_revenue,
        CASE
            WHEN activity.month = first_month.first_observed_month
                THEN 'First-observed customers'
            ELSE 'Returning customers'
        END AS customer_type
    FROM monthly_customer_activity AS activity
    INNER JOIN customer_first_month AS first_month
        ON activity.CustomerID = first_month.CustomerID
),

monthly_summary AS (
    SELECT
        month,
        customer_type,
        COUNT(*) AS customers,
        SUM(order_count) AS orders,
        SUM(customer_revenue) AS revenue
    FROM classified_activity
    GROUP BY month, customer_type
)

SELECT
    month,
    customer_type,
    customers,
    orders,
    ROUND(revenue, 2) AS revenue,
    ROUND(
        orders * 100.0 /
        SUM(orders) OVER (PARTITION BY month),
        2
    ) AS order_share_pct,
    ROUND(
        revenue * 100.0 /
        SUM(revenue) OVER (PARTITION BY month),
        2
    ) AS revenue_share_pct
FROM monthly_summary
ORDER BY month, customer_type;
"""

monthly_customer_mix_sql = pd.read_sql_query(
    monthly_customer_mix_query,conn)

monthly_customer_mix_sql

,month,customer_type,customers,orders,revenue,order_share_pct,revenue_share_pct
0,2010-12,First-observed customers,885,1400,570422.73,100.00,100.00
1,2011-01,First-observed customers,417,470,292366.84,47.62,51.46
2,2011-01,Returning customers,324,517,275734.47,52.38,48.54
3,2011-02,First-observed customers,380,422,157700.59,42.33,35.35
4,2011-02,Returning customers,378,575,288384.33,57.67,64.65
5,2011-03,First-observed customers,452,500,199619.67,37.85,33.60
6,2011-03,Returning customers,522,821,394462.09,62.15,66.40
7,2011-04,First-observed customers,300,339,121809.05,29.50,26.01
8,2011-04,Returning customers,556,810,346565.28,70.50,73.99
9,2011-05,First-observed customers,284,328,123739.30,21.09,18.27


### Insight

Returning customers generated most monthly revenue from February onward. By November, they contributed 88.31% of revenue and 85.06% of orders, showing that later sales relied heavily on customers already observed earlier in the dataset.


## 3. Order Behaviour by Customer Type

How do average order value and basket size differ between one-time and repeat customers?

In [14]:
customer_order_behaviour_query = """
WITH customer_frequency AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT InvoiceNo) AS total_orders
    FROM online_retail
    GROUP BY CustomerID
),

customer_types AS (
    SELECT
        CustomerID,
        CASE
            WHEN total_orders = 1 THEN 'One-time customers'
            ELSE 'Repeat customers'
        END AS customer_type
    FROM customer_frequency
),

order_summary AS (
    SELECT
        CustomerID,
        InvoiceNo,
        SUM(Total_price) AS order_value,
        SUM(Quantity) AS units,
        COUNT(DISTINCT StockCode) AS distinct_products
    FROM online_retail
    GROUP BY CustomerID, InvoiceNo)

SELECT
    customer_type,
    COUNT(DISTINCT orders.CustomerID) AS customers,
    COUNT(*) AS orders,
    ROUND(AVG(order_value), 2) AS average_order_value,
    ROUND(AVG(units), 2) AS average_units_per_order,
    ROUND(AVG(distinct_products), 2)
        AS average_distinct_products_per_order
FROM order_summary AS orders
INNER JOIN customer_types AS types
    ON orders.CustomerID = types.CustomerID
GROUP BY customer_type
ORDER BY average_order_value DESC;
"""

customer_order_behaviour_sql = pd.read_sql_query(
    customer_order_behaviour_query,conn)

customer_order_behaviour_sql

,customer_type,customers,orders,average_order_value,average_units_per_order,average_distinct_products_per_order
0,Repeat customers,2845,17039,485.55,278.68,20.86
1,One-time customers,1493,1493,411.25,270.29,21.68


### Insight

Repeat customers spent more per order, averaging 485.55 compared with 411.25 for one-time customers. Basket sizes were similar, so the difference may be linked to slightly higher quantities or the products purchased.


## 4. Order Behaviour Across RFM Segments

How do purchasing patterns differ across the RFM customer segments?

In [15]:
rfm_order_behaviour_query = """
WITH order_summary AS (
    SELECT
        CustomerID,
        InvoiceNo,
        SUM(Total_price) AS order_value,
        SUM(Quantity) AS units,
        COUNT(DISTINCT StockCode) AS distinct_products
    FROM online_retail
    GROUP BY CustomerID, InvoiceNo
)

SELECT
    segments.Segment,
    COUNT(DISTINCT orders.CustomerID) AS customers,
    COUNT(*) AS orders,
    
    ROUND(AVG(orders.order_value), 2)
        AS average_order_value,
    ROUND(AVG(orders.units), 2)
        AS average_units_per_order,
    ROUND(AVG(orders.distinct_products), 2)
        AS average_distinct_products_per_order
FROM order_summary AS orders
INNER JOIN rfm_segments AS segments
    ON orders.CustomerID = segments.CustomerID
GROUP BY segments.Segment
ORDER BY average_order_value DESC;
"""

rfm_order_behaviour_sql = pd.read_sql_query(
    rfm_order_behaviour_query,conn)

rfm_order_behaviour_sql

,Segment,customers,orders,average_order_value,average_units_per_order,average_distinct_products_per_order
0,High-Value At Risk,715,2515,544.76,317.70,21.50
1,High-Value Active Customers,791,9013,542.86,310.07,21.45
2,Potential Loyalists,739,1111,483.90,280.14,23.88
3,Loyal Customers,658,3831,431.56,254.83,21.47
4,At Risk,554,880,211.29,147.31,16.07
5,Inactive Customers,881,1182,209.36,119.49,14.82


### Insight

High-Value At Risk customers had the highest average order value at 544.76 and the highest average number of units per order. This suggests that they still place valuable orders when they purchase, making them a relevant group for re-engagement.


## 5. Products Purchased by Repeat Customers.

Which products are purchased by the largest number of repeat customers?

* Products are ranked by distinct customer count instead of revenue alone so that one unusually large order does not dominate the result. Non-product charges such as postage and bank fees are excluded.



In [16]:
repeat_customer_products_query = """
WITH customer_frequency AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT InvoiceNo) AS total_orders
    FROM online_retail
    GROUP BY CustomerID
),

repeat_customers AS (
    SELECT CustomerID
    FROM customer_frequency
    WHERE total_orders > 1
),

product_summary AS (
    SELECT
        transactions.StockCode,
        MAX(transactions.Description) AS Description,
        COUNT(DISTINCT transactions.CustomerID)
            AS repeat_customers,
        COUNT(DISTINCT transactions.InvoiceNo)
            AS repeat_orders,
        SUM(transactions.Quantity) AS units_sold,
        SUM(transactions.Total_price) AS revenue
    FROM online_retail AS transactions
    INNER JOIN repeat_customers
        ON transactions.CustomerID = repeat_customers.CustomerID
    WHERE transactions.StockCode NOT IN (
        'POST',
        'DOT',
        'M',
        'C2',
        'D',
        'S',
        'BANK CHARGES',
        'CRUK'
    )
    GROUP BY transactions.StockCode
),

ranked_products AS (
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY repeat_customers DESC, revenue DESC
        ) AS product_rank,
        StockCode,
        Description,
        repeat_customers,
        repeat_orders,
        units_sold,
        revenue
    FROM product_summary
)

SELECT
    product_rank,
    StockCode,
    Description,
    repeat_customers,
    repeat_orders,
    units_sold,
    ROUND(revenue, 2) AS revenue
FROM ranked_products
WHERE product_rank <= 10
ORDER BY product_rank;
"""

repeat_customer_products_sql = pd.read_sql_query(
    repeat_customer_products_query,
    conn)

repeat_customer_products_sql

,product_rank,StockCode,Description,repeat_customers,repeat_orders,units_sold,revenue
0,1,22423,REGENCY CAKESTAND 3 TIER,741,1563,11766,135081.55
1,2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,698,1820,35229,96241.35
2,3,47566,PARTY BUNTING,605,1276,14547,65375.93
3,4,84879,ASSORTED COLOUR BIRD ORNAMENT,565,1262,33724,53850.52
4,5,85099B,JUMBO BAG RED RETROSPOT,557,1522,44428,81936.20
5,6,21212,PACK OF 72 RETROSPOT CAKE CASES,554,948,31351,15246.83
6,7,22720,SET OF 3 CAKE TINS PANTRY DESIGN,537,1043,6523,31005.25
7,8,22086,PAPER CHAIN KIT 50'S CHRISTMAS,528,895,14690,40070.18
8,9,23298,SPOTTY BUNTING,517,953,7576,35139.40
9,10,22457,NATURAL SLATE HEART CHALKBOARD,497,877,7915,22578.77


### Insight

The Regency Cakestand was purchased by the largest number of repeat customers and generated 135,081.55 in revenue. Some widely purchased products produced much less revenue, showing that product popularity and revenue contribution should be evaluated separately.




## SQL Findings

* Returning customers became the main source of monthly sales. In November, they contributed 88.31% of revenue and 85.06% of orders.
* Repeat customers spent more per order than one-time customers, while their average basket sizes remained similar.
* High-Value At Risk customers recorded the highest average order value, making them an important group for re-engagement.
* The Regency Cakestand reached the largest number of repeat customers, but the results also showed that popular products are not always the highest-revenue products.

## Limitations

* First-observed customers cannot be confirmed as genuinely new because earlier purchase history is unavailable.
* Average order and basket values may be influenced by unusually large wholesale orders.
* RFM segments describe customer behaviour at one point in time and do not predict future purchases.
* Product results include positive purchases only because returns and cancellations were removed during cleaning.


In [17]:
conn.close()